<a href="https://colab.research.google.com/github/RohanT17/multimodal-cancer-classification/blob/main/Part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "transformers<4.46" "tf-keras"

In [2]:
"""
Late Fusion: Tabular (clinical+pathological+blood) + Text (surgery reports)
============================================================================

Strategy:
  1. Train tabular model (your existing text.py architecture) -> probabilities
  2. Train text model (BERT/ClinicalBERT on surgery reports) -> probabilities
  3. Combine the two probability distributions:
       Option A: simple weighted average (no learning)
       Option B: small meta-classifier on stacked probabilities (learned)

Targets: hpv_association_p16, primary_tumor_site, grading
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, TFAutoModel

# ============================================================
# CONFIG
# ============================================================
CLINICAL_PATH = "StructuredData/clinical_data.json"
PATHOLOGICAL_PATH = "StructuredData/pathological_data.json"
BLOOD_PATH = "StructuredData/blood_data.json"
TEXT_DIR = "TextData/reports/"   # one .txt per patient_id, or adapt loader

# Pretrained text encoder. ClinicalBERT works well for medical reports;
# fall back to "bert-base-uncased" if you can't download the clinical one.
TEXT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_TEXT_LEN = 256

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10

TARGETS = ["hpv_association_p16", "primary_tumor_site", "grading"]

# Same leaky columns you already identified
LEAKY_COLS = [
    "pT_stage", "pN_stage", "histologic_type", "perinodal_invasion",
    "lymphovascular_invasion_L", "vascular_invasion_V",
    "perineural_invasion_Pn", "resection_status",
    "resection_status_carcinoma_in_situ", "carcinoma_in_situ",
]

LOSS_WEIGHTS = {"hpv_out": 2.0, "site_out": 0.8, "grade_out": 1.2}


# ============================================================
# 1. DATA LOADING
# ============================================================
def load_json_to_df(path: str) -> pd.DataFrame:
    with open(path) as f:
        return pd.json_normalize(json.load(f))


def load_structured_data() -> pd.DataFrame:
    """Merge clinical + pathological + blood (pivoted) on patient_id."""
    clinical = load_json_to_df(CLINICAL_PATH)
    pathological = load_json_to_df(PATHOLOGICAL_PATH)
    blood_raw = load_json_to_df(BLOOD_PATH)

    blood_raw.columns = blood_raw.columns.str.strip()
    blood = blood_raw.pivot_table(
        index="patient_id", columns="analyte_name",
        values="value", aggfunc="mean",
    ).reset_index()
    blood.columns.name = None

    clinical.columns = clinical.columns.str.strip()
    pathological.columns = pathological.columns.str.strip()

    df = (
        clinical
        .merge(pathological, on="patient_id", how="inner")
        .merge(blood, on="patient_id", how="inner")
    )
    return df.dropna(subset=TARGETS)


def load_text_for_patient(patient_id: str, text_dir: str = TEXT_DIR) -> str:
    """
    Load surgery report text for a single patient.
    Adjust this to match HANCOCK's actual text file naming/structure.
    Common patterns:
      - {text_dir}/{patient_id}.txt
      - {text_dir}/{patient_id}/surgery_report.txt
      - one big JSON keyed by patient_id
    """
    candidates = [
        Path(text_dir) / f"{patient_id}.txt",
        Path(text_dir) / patient_id / "surgery_report.txt",
    ]
    for p in candidates:
        if p.exists():
            return p.read_text(encoding="utf-8", errors="ignore")
    return ""  # missing -> empty string (model can still tokenize)


def attach_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["report_text"] = df["patient_id"].apply(load_text_for_patient)
    return df


# ============================================================
# 2. TABULAR MODEL (same shape as your text.py)
# ============================================================
def build_tabular_model(input_dim, n_hpv, n_site, n_grade):
    inp = tf.keras.Input(shape=(input_dim,), name="tabular_input")

    x = tf.keras.layers.Dense(64, activation="relu")(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)

    hpv = tf.keras.layers.Dense(n_hpv, activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(n_site, activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(n_grade, activation="softmax", name="grade_out")(x)

    return tf.keras.Model(inp, [hpv, site, grade], name="tabular_model")


# ============================================================
# 3. TEXT MODEL (transformer encoder + multi-task heads)
# ============================================================
def tokenize_texts(texts, tokenizer, max_len=MAX_TEXT_LEN):
    enc = tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="np",
    )
    return {
        "input_ids": enc["input_ids"].astype(np.int32),
        "attention_mask": enc["attention_mask"].astype(np.int32),
    }


def build_text_model(n_hpv, n_site, n_grade, freeze_backbone=True):
    """Transformer-encoded surgery reports -> 3 task heads."""
    backbone = TFAutoModel.from_pretrained(TEXT_MODEL_NAME)
    backbone.trainable = not freeze_backbone

    input_ids = tf.keras.Input(shape=(MAX_TEXT_LEN,), dtype=tf.int32, name="input_ids")
    attn_mask = tf.keras.Input(shape=(MAX_TEXT_LEN,), dtype=tf.int32, name="attention_mask")

    bert_out = backbone(input_ids=input_ids, attention_mask=attn_mask)
    pooled = bert_out.last_hidden_state[:, 0, :]   # [CLS] token

    x = tf.keras.layers.Dense(64, activation="relu")(pooled)
    x = tf.keras.layers.Dropout(0.4)(x)

    hpv = tf.keras.layers.Dense(n_hpv, activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(n_site, activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(n_grade, activation="softmax", name="grade_out")(x)

    return tf.keras.Model([input_ids, attn_mask], [hpv, site, grade], name="text_model")


# ============================================================
# 4. LATE FUSION
# ============================================================
def fuse_average(tab_probs, txt_probs, w_tab=0.5, w_txt=0.5):
    """Simple weighted average of probability vectors (no training)."""
    return {
        task: w_tab * tab_probs[task] + w_txt * txt_probs[task]
        for task in ("hpv_out", "site_out", "grade_out")
    }


def stack_probs(tab_probs, txt_probs):
    """Concat all per-task probabilities into one feature vector per sample."""
    return np.concatenate(
        [tab_probs["hpv_out"], tab_probs["site_out"], tab_probs["grade_out"],
         txt_probs["hpv_out"], txt_probs["site_out"], txt_probs["grade_out"]],
        axis=1,
    )


def build_meta_fusion_model(input_dim, n_hpv, n_site, n_grade):
    """Small MLP that learns to combine the two modalities' probabilities."""
    inp = tf.keras.Input(shape=(input_dim,), name="fusion_input")
    x = tf.keras.layers.Dense(64, activation="relu")(inp)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)

    hpv = tf.keras.layers.Dense(n_hpv, activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(n_site, activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(n_grade, activation="softmax", name="grade_out")(x)

    return tf.keras.Model(inp, [hpv, site, grade], name="late_fusion_meta")


# ============================================================
# 5. EVALUATION
# ============================================================
def evaluate_task(y_true, y_pred_probs, task_name, encoder):
    y_pred = np.argmax(y_pred_probs, axis=1)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n--- {task_name} ---  Macro F1: {macro_f1:.4f}")

    try:
        if y_pred_probs.shape[1] == 2:
            auc = roc_auc_score(y_true, y_pred_probs[:, 1])
        else:
            auc = roc_auc_score(y_true, y_pred_probs,
                                multi_class="ovr", average="macro")
        print(f"  AUC-ROC: {auc:.4f}")
    except ValueError as e:
        print(f"  AUC-ROC: skipped ({e})")

    print(classification_report(
        y_true, y_pred,
        target_names=[str(c) for c in encoder.classes_],
        zero_division=0,
    ))
    return macro_f1


def evaluate_all(name, probs, y_true, encoders):
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    evaluate_task(y_true["hpv_out"], probs["hpv_out"],
                  "HPV", encoders["hpv_association_p16"])
    evaluate_task(y_true["site_out"], probs["site_out"],
                  "Primary site", encoders["primary_tumor_site"])
    evaluate_task(y_true["grade_out"], probs["grade_out"],
                  "Grading", encoders["grading"])


# ============================================================
# 6. MAIN
# ============================================================
def main():
    # ---- Load + attach text ----
    df = load_structured_data()
    df = attach_text(df)
    print("Merged shape:", df.shape)

    # ---- Patient-level split ----
    pids = df["patient_id"].unique()
    train_pids, test_pids = train_test_split(
        pids, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_pids, val_pids = train_test_split(
        train_pids, test_size=VAL_SIZE, random_state=RANDOM_STATE)

    train_df = df[df["patient_id"].isin(train_pids)].reset_index(drop=True)
    val_df = df[df["patient_id"].isin(val_pids)].reset_index(drop=True)
    test_df = df[df["patient_id"].isin(test_pids)].reset_index(drop=True)

    # ---- Encode targets (fit on full df to capture all classes) ----
    encoders, y_full = {}, {}
    for col in TARGETS:
        le = LabelEncoder().fit(df[col].astype(str))
        encoders[col] = le
        y_full[col] = le.transform(df[col].astype(str))

    n_hpv = len(encoders["hpv_association_p16"].classes_)
    n_site = len(encoders["primary_tumor_site"].classes_)
    n_grade = len(encoders["grading"].classes_)

    def labels_for(sub_df):
        return {
            "hpv_out": encoders["hpv_association_p16"].transform(
                sub_df["hpv_association_p16"].astype(str)),
            "site_out": encoders["primary_tumor_site"].transform(
                sub_df["primary_tumor_site"].astype(str)),
            "grade_out": encoders["grading"].transform(
                sub_df["grading"].astype(str)),
        }

    y_train = labels_for(train_df)
    y_val = labels_for(val_df)
    y_test = labels_for(test_df)

    # ---- Tabular preprocessing ----
    feat_cols = [c for c in df.columns
                 if c not in TARGETS + ["patient_id", "report_text"] + LEAKY_COLS]

    Xtr_oh = pd.get_dummies(train_df[feat_cols], dummy_na=True)
    Xva_oh = pd.get_dummies(val_df[feat_cols], dummy_na=True)
    Xte_oh = pd.get_dummies(test_df[feat_cols], dummy_na=True)
    Xtr_oh, Xva_oh = Xtr_oh.align(Xva_oh, join="left", axis=1, fill_value=0)
    Xtr_oh, Xte_oh = Xtr_oh.align(Xte_oh, join="left", axis=1, fill_value=0)

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr_tab = sc.fit_transform(imp.fit_transform(Xtr_oh)).astype(np.float32)
    Xva_tab = sc.transform(imp.transform(Xva_oh)).astype(np.float32)
    Xte_tab = sc.transform(imp.transform(Xte_oh)).astype(np.float32)

    # ---- Text tokenization ----
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
    Xtr_txt = tokenize_texts(train_df["report_text"], tokenizer)
    Xva_txt = tokenize_texts(val_df["report_text"], tokenizer)
    Xte_txt = tokenize_texts(test_df["report_text"], tokenizer)

    # ---- Sample weights for HPV imbalance ----
    cls = np.unique(y_train["hpv_out"])
    w = compute_class_weight("balanced", classes=cls, y=y_train["hpv_out"])
    w_map = dict(zip(cls, w))
    sw = np.array([w_map[c] for c in y_train["hpv_out"]])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=PATIENCE // 2, min_lr=1e-6, verbose=1),
    ]
    compile_kwargs = dict(
        optimizer="adam",
        loss={k: "sparse_categorical_crossentropy"
              for k in ("hpv_out", "site_out", "grade_out")},
        loss_weights=LOSS_WEIGHTS,
        metrics={k: ["accuracy"]
                 for k in ("hpv_out", "site_out", "grade_out")},
    )

    # ============================================================
    # STEP A — Tabular baseline
    # ============================================================
    print("\n" + "=" * 60 + "\nTraining TABULAR baseline\n" + "=" * 60)
    tab_model = build_tabular_model(Xtr_tab.shape[1], n_hpv, n_site, n_grade)
    tab_model.compile(**compile_kwargs)
    tab_model.fit(
        Xtr_tab,
        [y_train["hpv_out"], y_train["site_out"], y_train["grade_out"]],
        validation_data=(Xva_tab,
                         [y_val["hpv_out"], y_val["site_out"], y_val["grade_out"]]),
        sample_weight=sw,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1,
    )

    # ============================================================
    # STEP B — Text baseline
    # ============================================================
    print("\n" + "=" * 60 + "\nTraining TEXT baseline\n" + "=" * 60)
    txt_model = build_text_model(n_hpv, n_site, n_grade, freeze_backbone=True)
    txt_model.compile(
        optimizer=tf.keras.optimizers.Adam(2e-5),
        loss=compile_kwargs["loss"],
        loss_weights=compile_kwargs["loss_weights"],
        metrics=compile_kwargs["metrics"],
    )
    txt_model.fit(
        [Xtr_txt["input_ids"], Xtr_txt["attention_mask"]],
        [y_train["hpv_out"], y_train["site_out"], y_train["grade_out"]],
        validation_data=(
            [Xva_txt["input_ids"], Xva_txt["attention_mask"]],
            [y_val["hpv_out"], y_val["site_out"], y_val["grade_out"]],
        ),
        sample_weight=sw,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1,
    )

    # ============================================================
    # STEP C — Generate probabilities on val + test
    # ============================================================
    print("\nGenerating modality probabilities for fusion...")

    def predict_tab(X):
        preds = tab_model.predict(X, verbose=0)
        return {"hpv_out": preds[0], "site_out": preds[1], "grade_out": preds[2]}

    def predict_txt(X):
        preds = txt_model.predict(
            [X["input_ids"], X["attention_mask"]], verbose=0)
        return {"hpv_out": preds[0], "site_out": preds[1], "grade_out": preds[2]}

    tab_val, tab_test = predict_tab(Xva_tab), predict_tab(Xte_tab)
    txt_val, txt_test = predict_txt(Xva_txt), predict_txt(Xte_txt)

    # Unimodal results (baselines)
    evaluate_all("TABULAR ONLY (test)", tab_test, y_test, encoders)
    evaluate_all("TEXT ONLY (test)", txt_test, y_test, encoders)

    # ============================================================
    # STEP D — Late fusion: simple average
    # ============================================================
    fused_avg = fuse_average(tab_test, txt_test, w_tab=0.5, w_txt=0.5)
    evaluate_all("LATE FUSION — weighted average (test)",
                 fused_avg, y_test, encoders)

    # ============================================================
    # STEP E — Late fusion: learned meta-classifier
    # ============================================================
    print("\n" + "=" * 60 + "\nTraining META-fusion classifier\n" + "=" * 60)
    Xv_meta = stack_probs(tab_val, txt_val)
    Xt_meta = stack_probs(tab_test, txt_test)

    meta = build_meta_fusion_model(Xv_meta.shape[1], n_hpv, n_site, n_grade)
    meta.compile(**compile_kwargs)
    meta.fit(
        Xv_meta,
        [y_val["hpv_out"], y_val["site_out"], y_val["grade_out"]],
        epochs=50, batch_size=BATCH_SIZE,
        validation_split=0.2, verbose=1,
    )

    meta_preds = meta.predict(Xt_meta, verbose=0)
    fused_meta = {"hpv_out": meta_preds[0],
                  "site_out": meta_preds[1],
                  "grade_out": meta_preds[2]}
    evaluate_all("LATE FUSION — meta-classifier (test)",
                 fused_meta, y_test, encoders)

    tab_model.save("tabular_model.keras")
    txt_model.save("text_model.keras")
    meta.save("late_fusion_meta.keras")
    print("\nAll models saved.")


if __name__ == "__main__":
    main()

FileNotFoundError: [Errno 2] No such file or directory: 'StructuredData/clinical_data.json'

In [ ]:
"""
Attention-Based Fusion: Tabular (clinical+pathological+blood) + Text (surgery reports)
========================================================================================

Strategy:
  - Each modality has its own encoder that produces an embedding (vector, not probabilities)
  - An attention mechanism learns per-sample weights over the modality embeddings
  - The attention-weighted combined embedding feeds into the multi-task heads
  - Trained END-TO-END (unlike late fusion, which trains modalities separately)

Why this is "more advanced" than late fusion:
  - Late fusion: fixed combination of two independently-trained models' outputs
  - Attention fusion: model dynamically decides "for THIS patient, trust the
    text report 70% and the tabular data 30%" based on the inputs themselves

Targets: hpv_association_p16, primary_tumor_site, grading
"""

import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, TFAutoModel

# ============================================================
# CONFIG
# ============================================================
CLINICAL_PATH = "StructuredData/clinical_data.json"
PATHOLOGICAL_PATH = "StructuredData/pathological_data.json"
BLOOD_PATH = "StructuredData/blood_data.json"
TEXT_DIR = "TextData/reports/"

TEXT_MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_TEXT_LEN = 256
EMBED_DIM = 64   # dimension all modality embeddings get projected to

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2
BATCH_SIZE = 16
MAX_EPOCHS = 50
PATIENCE = 10

TARGETS = ["hpv_association_p16", "primary_tumor_site", "grading"]

LEAKY_COLS = [
    "pT_stage", "pN_stage", "histologic_type", "perinodal_invasion",
    "lymphovascular_invasion_L", "vascular_invasion_V",
    "perineural_invasion_Pn", "resection_status",
    "resection_status_carcinoma_in_situ", "carcinoma_in_situ",
]

LOSS_WEIGHTS = {"hpv_out": 2.0, "site_out": 0.8, "grade_out": 1.2}


# ============================================================
# 1. DATA LOADING (same helpers as late_fusion.py)
# ============================================================
def load_json_to_df(path: str) -> pd.DataFrame:
    with open(path) as f:
        return pd.json_normalize(json.load(f))


def load_structured_data() -> pd.DataFrame:
    clinical = load_json_to_df(CLINICAL_PATH)
    pathological = load_json_to_df(PATHOLOGICAL_PATH)
    blood_raw = load_json_to_df(BLOOD_PATH)

    blood_raw.columns = blood_raw.columns.str.strip()
    blood = blood_raw.pivot_table(
        index="patient_id", columns="analyte_name",
        values="value", aggfunc="mean",
    ).reset_index()
    blood.columns.name = None

    clinical.columns = clinical.columns.str.strip()
    pathological.columns = pathological.columns.str.strip()

    df = (
        clinical
        .merge(pathological, on="patient_id", how="inner")
        .merge(blood, on="patient_id", how="inner")
    )
    return df.dropna(subset=TARGETS)


def load_text_for_patient(patient_id: str, text_dir: str = TEXT_DIR) -> str:
    candidates = [
        Path(text_dir) / f"{patient_id}.txt",
        Path(text_dir) / patient_id / "surgery_report.txt",
    ]
    for p in candidates:
        if p.exists():
            return p.read_text(encoding="utf-8", errors="ignore")
    return ""


def attach_text(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["report_text"] = df["patient_id"].apply(load_text_for_patient)
    return df


def tokenize_texts(texts, tokenizer, max_len=MAX_TEXT_LEN):
    enc = tokenizer(
        list(texts),
        padding="max_length", truncation=True,
        max_length=max_len, return_tensors="np",
    )
    return {
        "input_ids": enc["input_ids"].astype(np.int32),
        "attention_mask": enc["attention_mask"].astype(np.int32),
    }


# ============================================================
# 2. MODALITY ENCODERS (produce embeddings, NOT probabilities)
# ============================================================
def build_tabular_encoder(input_dim, embed_dim=EMBED_DIM):
    """Tabular -> embedding vector of shape (embed_dim,)."""
    inp = tf.keras.Input(shape=(input_dim,), name="tabular_input")
    x = tf.keras.layers.Dense(64, activation="relu")(inp)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    out = tf.keras.layers.Dense(embed_dim, activation="relu",
                                name="tabular_embedding")(x)
    return tf.keras.Model(inp, out, name="tabular_encoder")


def build_text_encoder(embed_dim=EMBED_DIM, freeze_backbone=True):
    """Surgery report tokens -> embedding vector of shape (embed_dim,)."""
    backbone = TFAutoModel.from_pretrained(TEXT_MODEL_NAME)
    backbone.trainable = not freeze_backbone

    input_ids = tf.keras.Input(shape=(MAX_TEXT_LEN,), dtype=tf.int32, name="input_ids")
    attn_mask = tf.keras.Input(shape=(MAX_TEXT_LEN,), dtype=tf.int32, name="attention_mask")

    bert_out = backbone(input_ids=input_ids, attention_mask=attn_mask)
    pooled = bert_out.last_hidden_state[:, 0, :]    # [CLS]

    x = tf.keras.layers.Dropout(0.3)(pooled)
    out = tf.keras.layers.Dense(embed_dim, activation="relu",
                                name="text_embedding")(x)
    return tf.keras.Model([input_ids, attn_mask], out, name="text_encoder")


# ============================================================
# 3. ATTENTION FUSION LAYER
# ============================================================
class ModalityAttention(tf.keras.layers.Layer):
    """
    Given a stack of modality embeddings, shape (batch, n_modalities, embed_dim),
    learn per-sample attention weights and return the weighted sum.

    Mechanism (simple additive / Bahdanau-style):
        score_m = v^T tanh(W e_m)
        alpha_m = softmax(score_m) over modalities
        out     = sum_m alpha_m * e_m

    Also exposes the attention weights so we can inspect which modality the
    model relied on for each sample (great for the discussion/analysis section).
    """
    def __init__(self, attention_dim=32, **kwargs):
        super().__init__(**kwargs)
        self.attention_dim = attention_dim

    def build(self, input_shape):
        # input_shape: (batch, n_modalities, embed_dim)
        embed_dim = input_shape[-1]
        self.W = self.add_weight(
            shape=(embed_dim, self.attention_dim),
            initializer="glorot_uniform", name="W")
        self.v = self.add_weight(
            shape=(self.attention_dim, 1),
            initializer="glorot_uniform", name="v")
        super().build(input_shape)

    def call(self, embeddings):
        # embeddings: (batch, n_modalities, embed_dim)
        scores = tf.tanh(tf.matmul(embeddings, self.W))   # (B, M, A)
        scores = tf.matmul(scores, self.v)                # (B, M, 1)
        weights = tf.nn.softmax(scores, axis=1)           # (B, M, 1)
        fused = tf.reduce_sum(weights * embeddings, axis=1)   # (B, embed_dim)
        return fused, tf.squeeze(weights, axis=-1)        # also return weights


# ============================================================
# 4. END-TO-END ATTENTION FUSION MODEL
# ============================================================
def build_attention_fusion_model(tab_dim, n_hpv, n_site, n_grade,
                                 embed_dim=EMBED_DIM, freeze_text=True):
    tab_enc = build_tabular_encoder(tab_dim, embed_dim)
    txt_enc = build_text_encoder(embed_dim, freeze_backbone=freeze_text)

    tab_input = tab_enc.input
    txt_input_ids = txt_enc.input[0]
    txt_attn_mask = txt_enc.input[1]

    tab_emb = tab_enc(tab_input)                       # (B, embed_dim)
    txt_emb = txt_enc([txt_input_ids, txt_attn_mask])  # (B, embed_dim)

    # Stack embeddings into shape (B, n_modalities=2, embed_dim)
    stacked = tf.keras.layers.Lambda(
        lambda t: tf.stack(t, axis=1),
        name="stack_modalities",
    )([tab_emb, txt_emb])

    fused, attn_weights = ModalityAttention(attention_dim=32,
                                            name="modality_attention")(stacked)

    x = tf.keras.layers.Dense(64, activation="relu")(fused)
    x = tf.keras.layers.Dropout(0.3)(x)

    hpv = tf.keras.layers.Dense(n_hpv, activation="softmax", name="hpv_out")(x)
    site = tf.keras.layers.Dense(n_site, activation="softmax", name="site_out")(x)
    grade = tf.keras.layers.Dense(n_grade, activation="softmax", name="grade_out")(x)

    # Expose attention weights as an extra (non-loss) output for inspection.
    # We give it a name so we can pull it out at predict time.
    attn_out = tf.keras.layers.Lambda(lambda t: t, name="attn_weights")(attn_weights)

    model = tf.keras.Model(
        inputs=[tab_input, txt_input_ids, txt_attn_mask],
        outputs=[hpv, site, grade, attn_out],
        name="attention_fusion",
    )
    return model


# ============================================================
# 5. EVALUATION HELPERS
# ============================================================
def evaluate_task(y_true, y_pred_probs, task_name, encoder):
    y_pred = np.argmax(y_pred_probs, axis=1)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n--- {task_name} ---  Macro F1: {macro_f1:.4f}")

    try:
        if y_pred_probs.shape[1] == 2:
            auc = roc_auc_score(y_true, y_pred_probs[:, 1])
        else:
            auc = roc_auc_score(y_true, y_pred_probs,
                                multi_class="ovr", average="macro")
        print(f"  AUC-ROC: {auc:.4f}")
    except ValueError as e:
        print(f"  AUC-ROC: skipped ({e})")

    print(classification_report(
        y_true, y_pred,
        target_names=[str(c) for c in encoder.classes_],
        zero_division=0,
    ))
    return macro_f1


# ============================================================
# 6. MAIN
# ============================================================
def main():
    df = load_structured_data()
    df = attach_text(df)
    print("Merged shape:", df.shape)

    # Patient-level split
    pids = df["patient_id"].unique()
    train_pids, test_pids = train_test_split(
        pids, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    train_pids, val_pids = train_test_split(
        train_pids, test_size=VAL_SIZE, random_state=RANDOM_STATE)

    train_df = df[df["patient_id"].isin(train_pids)].reset_index(drop=True)
    val_df = df[df["patient_id"].isin(val_pids)].reset_index(drop=True)
    test_df = df[df["patient_id"].isin(test_pids)].reset_index(drop=True)

    encoders = {}
    for col in TARGETS:
        encoders[col] = LabelEncoder().fit(df[col].astype(str))

    n_hpv = len(encoders["hpv_association_p16"].classes_)
    n_site = len(encoders["primary_tumor_site"].classes_)
    n_grade = len(encoders["grading"].classes_)

    def labels_for(sub_df):
        return {
            "hpv_out": encoders["hpv_association_p16"].transform(
                sub_df["hpv_association_p16"].astype(str)),
            "site_out": encoders["primary_tumor_site"].transform(
                sub_df["primary_tumor_site"].astype(str)),
            "grade_out": encoders["grading"].transform(
                sub_df["grading"].astype(str)),
        }

    y_train = labels_for(train_df)
    y_val = labels_for(val_df)
    y_test = labels_for(test_df)

    # Tabular preprocessing
    feat_cols = [c for c in df.columns
                 if c not in TARGETS + ["patient_id", "report_text"] + LEAKY_COLS]
    Xtr_oh = pd.get_dummies(train_df[feat_cols], dummy_na=True)
    Xva_oh = pd.get_dummies(val_df[feat_cols], dummy_na=True)
    Xte_oh = pd.get_dummies(test_df[feat_cols], dummy_na=True)
    Xtr_oh, Xva_oh = Xtr_oh.align(Xva_oh, join="left", axis=1, fill_value=0)
    Xtr_oh, Xte_oh = Xtr_oh.align(Xte_oh, join="left", axis=1, fill_value=0)

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    Xtr_tab = sc.fit_transform(imp.fit_transform(Xtr_oh)).astype(np.float32)
    Xva_tab = sc.transform(imp.transform(Xva_oh)).astype(np.float32)
    Xte_tab = sc.transform(imp.transform(Xte_oh)).astype(np.float32)

    # Text tokenization
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
    Xtr_txt = tokenize_texts(train_df["report_text"], tokenizer)
    Xva_txt = tokenize_texts(val_df["report_text"], tokenizer)
    Xte_txt = tokenize_texts(test_df["report_text"], tokenizer)

    # HPV class weights
    cls = np.unique(y_train["hpv_out"])
    w = compute_class_weight("balanced", classes=cls, y=y_train["hpv_out"])
    w_map = dict(zip(cls, w))
    sw = np.array([w_map[c] for c in y_train["hpv_out"]])

    # ============================================================
    # Build & train end-to-end attention fusion model
    # ============================================================
    model = build_attention_fusion_model(
        tab_dim=Xtr_tab.shape[1],
        n_hpv=n_hpv, n_site=n_site, n_grade=n_grade,
        freeze_text=True,
    )
    model.summary()

    # Note: we provide losses ONLY for the 3 task outputs; attn_weights is
    # output for inspection only (no target, no loss).
    model.compile(
        optimizer=tf.keras.optimizers.Adam(2e-5),
        loss={
            "hpv_out": "sparse_categorical_crossentropy",
            "site_out": "sparse_categorical_crossentropy",
            "grade_out": "sparse_categorical_crossentropy",
            # No loss on attn_weights:
            "attn_weights": None,
        },
        loss_weights=LOSS_WEIGHTS,
        metrics={
            "hpv_out": ["accuracy"],
            "site_out": ["accuracy"],
            "grade_out": ["accuracy"],
        },
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=PATIENCE // 2, min_lr=1e-6, verbose=1),
    ]

    print("\n" + "=" * 60 + "\nTraining ATTENTION fusion (end-to-end)\n" + "=" * 60)
    model.fit(
        x=[Xtr_tab, Xtr_txt["input_ids"], Xtr_txt["attention_mask"]],
        y={"hpv_out": y_train["hpv_out"],
           "site_out": y_train["site_out"],
           "grade_out": y_train["grade_out"]},
        validation_data=(
            [Xva_tab, Xva_txt["input_ids"], Xva_txt["attention_mask"]],
            {"hpv_out": y_val["hpv_out"],
             "site_out": y_val["site_out"],
             "grade_out": y_val["grade_out"]},
        ),
        sample_weight=sw,
        epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1,
    )

    # ============================================================
    # Evaluate on test
    # ============================================================
    hpv_p, site_p, grade_p, attn_w = model.predict(
        [Xte_tab, Xte_txt["input_ids"], Xte_txt["attention_mask"]],
        verbose=0,
    )

    print("\n" + "=" * 60 + "\nATTENTION FUSION — test results\n" + "=" * 60)
    evaluate_task(y_test["hpv_out"], hpv_p, "HPV",
                  encoders["hpv_association_p16"])
    evaluate_task(y_test["site_out"], site_p, "Primary site",
                  encoders["primary_tumor_site"])
    evaluate_task(y_test["grade_out"], grade_p, "Grading",
                  encoders["grading"])

    # Attention analysis (great for the writeup): which modality did the
    # model lean on, on average?
    mean_attn = attn_w.mean(axis=0)
    print(f"\nMean attention weights over test set:")
    print(f"  tabular: {mean_attn[0]:.3f}")
    print(f"  text   : {mean_attn[1]:.3f}")

    model.save("attention_fusion.keras")
    print("\nModel saved to attention_fusion.keras")


if __name__ == "__main__":
    main()